<a href="https://colab.research.google.com/github/Susan-sapkota-98/Nepali-asr-evaluation/blob/main/11_sound_folding_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 25.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [jiwer]


In [3]:
from pathlib import Path

import jiwer
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')
folder = Path("/content/drive/MyDrive/nepali-asr-evaluation/rating_200")

Mounted at /content/drive


In [4]:
my_ratings = pd.read_csv(folder / "my_ratings_round2.csv", dtype=str, keep_default_na=False)
private = pd.read_csv(folder / "rating_rows_private_round2.csv", dtype={"file_id": str}, keep_default_na=False)
joined = my_ratings.merge(private, on="file_id")

rated = joined[joined["rating"].isin(["0", "1"])].copy()
rated["rating"] = rated["rating"].astype(int)
print(len(rated), "rated rows")

101 rated rows


In [5]:
digit_map = str.maketrans("०१२३४५६७८९", "0123456789")


def fold_text(text):
    for mark in "।,.?!;:\"'":
        text = text.replace(mark, " ")
    text = text.translate(digit_map)          # Nepali digits to normal digits
    text = text.replace("ङ्", "ं")             # ङ् and ं are treated the same
    text = text.replace("ँ", "ं")              # chandrabindu and ं are the same
    text = text.replace("ी", "ि").replace("ू", "ु")    # long vowel to short
    text = text.replace("श", "स").replace("ष", "स")    # श, ष, स are the same
    text = text.replace("्", "")               # remove halant
    return text.replace(" ", "")              # remove spaces


print(fold_text("सङ्घर्ष गर्नमा"))
print(fold_text("संघर्ष गर्नमा"))

संघरसगरनमा
संघरसगरनमा


In [6]:
rated["folded_cer"] = [jiwer.cer(fold_text(c), fold_text(m)) for c, m in zip(rated["correct_text"], rated["model_text"])]
rated["forgiven"] = rated["folded_cer"] == 0

pd.crosstab(rated["rating"], rated["forgiven"])

forgiven,False,True
rating,,
0,56,3
1,23,19


In [7]:
rated.groupby("rating")[["wer", "cer", "folded_cer"]].mean().round(2)

,wer,cer,folded_cer
rating,,,
0,0.68,0.24,0.20
1,0.55,0.10,0.05


In [8]:
for score_name in ["wer", "cer", "folded_cer"]:
    print(score_name, round(rated["rating"].corr(rated[score_name], method="spearman"), 2))

wer -0.13
cer -0.48
folded_cer -0.62


In [9]:
accepted_left = rated[(rated["rating"] == 1) & (~rated["forgiven"])].sort_values("folded_cer", ascending=False)
print("accepted but still different:", len(accepted_left))
accepted_left[["correct_text", "model_text", "folded_cer"]].head(8)

accepted but still different: 23


,correct_text,model_text,folded_cer
69,जाने ४५ किलोमिटर,जाने पैनतालिस किलोमिटर,0.571429
36,गर्नु पर्थ्यो,गर्नु पर्थियो,0.111111
7,युद्धको सुरुआतको उल्लेख,युद्धको सुरुवातको उल्लेख,0.105263
85,उहाँ धौलागिरी अञ्चलबाट,उहाँ धलाग्री अञ्चलबाट,0.105263
89,आक्रमण भै रहन्छ,आक्रमण भइरहन्छ,0.090909
67,यो नदी समुन्द्रमा,यो नदी समुद्रमा,0.076923
17,पानीमा धेरै समय,पानीमा घेरै समय,0.076923
81,विराट मनका नेता,बिराटमनका नेता,0.076923


In [10]:
too_forgiving = rated[(rated["rating"] == 0) & (rated["forgiven"])]
print("rejected but forgiven:", len(too_forgiving))
too_forgiving[["correct_text", "model_text"]]

rejected but forgiven: 3


,correct_text,model_text
51,खोलाको सङ्गममा इन्द्रले,खोलाको सं्गममा इन्द्रले
68,हस्ता शब्दले हात,हस्ता शब्दले ह्ात
88,अर्थव्यवस्थाको ला,अर्थ व्यवस्था कोला
